In [ ]:
import sys,os,re
import numpy             as np
import matplotlib.pyplot as plt
import pandas            as pd
import seaborn           as sb

from source_code.galdist import galaxy_distribution

from itertools import product
from copy      import deepcopy
from time      import time

from scipy.interpolate import interp1d
from scipy.integrate   import trapz

import warnings
warnings.filterwarnings('ignore')

import matplotlib
from matplotlib import rc
from matplotlib.pyplot import cm
from matplotlib.colors import LogNorm

rc('text', usetex=True)
rc('font', family='serif')
matplotlib.rcParams.update({'font.size': 18})

red    = '#8e001c'
yellow = '#ffb302'

sidelegend = {'bbox_to_anchor': (1.04,0.5), 
              'loc': "center left",
              'frameon': False}

#sys.path.insert(0,'/Users/chiaradeleo/Desktop/camb_fork/GW-CAMB')
import camb
print(camb.__path__)

In [ ]:
#Euclid survey specifications, N_gw in [10^5-10^6] according to ET
galaxy_specs = {'fsky': 0.35, 
                'gal_per_arcmin': 30.,
                'sigma_eps': 0.3,  #sigma associated to noise for GC and WL
                'Nbin_ell': 20,
                'lmin': 10,
                'lmax': 1500}

GW_specs = {'fsky': 0.35, 
            'N_gw': 10**5, 
            'sigma_eps_gw': 0.005} #sigma associated to noise (d_L) for GW-WL

analysis_settings = {'Nbin_ell': 20,
                     'lmin': 10,
                     'lmax': 1500}

use_obs  = ['GC','WL', 'GWC', 'GWWL']

In [ ]:
lmin = np.log10(analysis_settings['lmin'])
lmax = np.log10(analysis_settings['lmax'])
N    = analysis_settings['Nbin_ell']

ell_lims = np.logspace(lmin,lmax,N) #creation of array-> N bin log spaced
ells     = np.array([int(ell) for ell in 0.5*(ell_lims[:-1]+ell_lims[1:])]) 
#evaluation of middle points of each bin 
deltas   = (ell_lims[1:]-ell_lims[:-1]) #evaluation of the amplitude of each bin

In [ ]:
fiducial = {'ombh2': 0.022445,
            'omch2': 0.1205579307,
            'ns': 0.96,
            'As': 2.12605e-09,
            'tau': 0.05,
            'H0': 67.,
            'w': -1.,
            'wa': 0.,
            'mnu': 0.06,
            'a0': - 0.007589,
            'a1' :  0.002008,
            'a2' : - 0.004127,
            'a3' :  0.002918,
            'a4' : -0.0006784,
            'omegab': 0.05,
            'sigma8': 0.84,
            'omegam' : 0.31,
            #'A_IA': 1.72,
            #'eta_IA': -0.41,
            'b0_poly': 0.830703,
            'b1_poly': 1.190547,
            'b2_poly': -0.928357,
            'b3_poly': 0.423292}
MG_params={'MG_flag': 0}
fiducial.update(MG_params)
bincolors = sb.color_palette('rainbow',10)

In [ ]:
fiducial_MG = {'ombh2': 0.022445,
            'omch2': 0.1205579307,
            'ns': 0.96,
            'As': 2.12605e-09,
            'tau': 0.05,
            'H0': 67.,
            'w': -1.,
            'wa': 0.,
            'mnu': 0.06,
            'a0': - 0.007589,
            'a1' :  0.002008,
            'a2' : - 0.004127,
            'a3' :  0.002918,
            'a4' : -0.0006784,
            'omegab': 0.05,
            'sigma8': 0.84,
            'omegam' : 0.31,
            'b0_poly': 0.830703,
            'b1_poly': 1.190547,
            'b2_poly': -0.928357,
            'b3_poly': 0.423292}
MG_params={'MG_flag': 1,
           'pure_MG_flag': 2,
           'musigma_par': 1,
           'DE_model': 0,
           'sigma0': -0.17,
           'mu0': -0.58}
fiducial_MG.update(MG_params)

In [ ]:
distributions = {}

In [ ]:
if 'GC' in use_obs:
    dist = galaxy_distribution(survey='Euclid-10')
    bin_lims = dist.galdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_gc = len(bin_lims)-1
    
    distributions['GC'] = {'dist': dist.galdict['binned_dist'],
                           'Nbins': Nbins_gc,
                           'zmean': bin_mids}
    
if 'WL' in use_obs:
    dist = galaxy_distribution(survey='Euclid-10')
    bin_lims = dist.galdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_wl = len(bin_lims)-1
    
    distributions['WL'] = {'dist': dist.galdict['binned_dist'],
                           'Nbins': Nbins_wl,
                           'zmean': bin_mids}
    

if 'GWC' in use_obs:
    from source_code.gwdist import gw_distribution
    gw_dist = gw_distribution(survey='ET-10')
   
    bin_lims = gw_dist.gwdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_gwc = len(bin_lims)-1
    
    distributions['GWC'] = {'dist': gw_dist.gwdict['binned_dist'],
                           'Nbins': Nbins_gwc,
                           'zmean': bin_mids}
if 'GWWL' in use_obs:
    from source_code.gwdist import gw_distribution
    gw_dist = gw_distribution(survey='ET-10')
   
    bin_lims = gw_dist.gwdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_gwl = len(bin_lims)-1
    
    distributions['GWWL'] = {'dist': gw_dist.gwdict['binned_dist'],
                           'Nbins': Nbins_gwl,
                           'zmean': bin_mids}

In [ ]:
zplot = np.linspace(0.001,4,100)
    
plt.figure()    
for i in range(1,distributions['GWC']['Nbins']):
    plt.plot(zplot,distributions['GWWL']['dist'][i](zplot),color=bincolors[i],label=r'Bin {}'.format(i+1))
    plt.plot(zplot,distributions['GWC']['dist'][i](zplot),color=bincolors[i],label=r'Bin {}'.format(i+1),marker = '*')
    
plt.xlabel(r'$z$')
plt.ylabel(r'$n_{gw}(z)$')
plt.legend(**sidelegend)
plt.title('Gravitational Waves distribution')

In [ ]:
from source_code.compute_obs_sources import get_obs
extra={}
settings={'camb_path': camb.__path__,
         'case': 'simple',
         'calculation': 'internal',
          'extra':extra}

In [ ]:
calc_obs = get_obs(fiducial,distributions,ells,settings,feedback=True)


In [ ]:
settings={'camb_path': camb.__path__,
         'case': 'simple',
         'calculation': 'CAMB',
          'extra':extra}
calc_obs2 = get_obs(fiducial,distributions,ells,settings,feedback=True)


In [ ]:

if len(use_obs)==1:
    Ncols=1
elif len(use_obs)==2:
    Ncols=3
elif len(use_obs)==3:
    Ncols=6
else:
    Ncols=6

In [ ]:
norm = ells*(ells+1)/(2*np.pi)
fig, axes = plt.subplots(ncols=Ncols, sharey=False, subplot_kw=dict(frameon=True),figsize=(12,4))
plt.suptitle('Fiducial Angular power spectra, old: (-), new: (*)',y=1.05)
axis_label_fontsize = 20
title_fontsize = 20
title_fontweight = 'bold'
if 'GC' in use_obs:
    axes[0].set_title('GCph',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_gc+1):
        axes[0].plot(ells,calc_obs.Cls['G{}xG{}'.format(i,i)]*norm,
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[0].plot(ells,calc_obs2.Cls['G{}xG{}'.format(i,i)]*norm, marker = '*',
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[0].set_xlabel(r'$\ell$')
    axes[0].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[0].set_xscale('log')
    axes[0].set_yscale('log')
    
if 'WL' in use_obs:
    axes[1].set_title('WL',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_wl+1):
        axes[1].plot(ells,calc_obs.Cls['L{}xL{}'.format(i,i)]*norm,
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[1].plot(ells,calc_obs2.Cls['L{}xL{}'.format(i,i)]*norm, marker = '*',
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[1].set_xlabel(r'$\ell$')
    axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')


if 'GWWL' in use_obs:
    axes[2].set_title('GW',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_gw+1):
        axes[2].plot(ells,calc_obs.Cls['W{}xW{}'.format(i,i)]*norm,
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
        
        axes[2].plot(ells,calc_obs2.Cls['WL{}xWL{}'.format(i,i)]*norm, marker = '*',
                     
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[2].set_xlabel(r'$\ell$')
    axes[2].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[2].set_xscale('log')
    axes[2].set_yscale('log')
    
if 'GC' in use_obs and 'WL' in use_obs:
    axes[3].set_title('GGL',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_gc+1):
        axes[3].plot(ells,calc_obs.Cls['G{}xL{}'.format(i,i)]*norm,
                    label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[3].plot(ells,calc_obs2.Cls['G{}xL{}'.format(i,i)]*norm, marker = '*',
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[3].set_xlabel(r'$\ell$')
    axes[3].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[3].set_xscale('log')


if 'GC' in use_obs and 'GWWL' in use_obs:
    axes[4].set_title('GCphxGW',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_gc+1):
        axes[4].plot(ells,calc_obs.Cls['G{}xW{}'.format(i,i)]*norm,
                    label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[4].plot(ells,calc_obs2.Cls['G{}xWL{}'.format(i,i)]*norm, marker = '*',
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[4].set_xlabel(r'$\ell$')
    axes[4].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[4].set_xscale('log')
   


if 'WL' in use_obs and 'GWWL' in use_obs:
    axes[5].set_title('WLxGW',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_gc+1):
        axes[5].plot(ells,calc_obs.Cls['L{}xW{}'.format(i,i)]*norm,
                    label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[5].plot(ells,calc_obs2.Cls['L{}xWL{}'.format(i,i)]*norm, marker = '*',
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[5].set_xlabel(r'$\ell$')
    axes[5].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[5].set_xscale('log')

axes[-1].legend(**sidelegend)
axes[0].set_ylabel(r'$\ell(\ell+1)/(2\pi)C_{ii}^{AB}(\ell)$', fontsize=axis_label_fontsize)
plt.subplots_adjust(hspace=1.2)
#plt.savefig('Cls_GWWL_old_new_test.png', bbox_inches='tight')
axes[-1].legend(**sidelegend);

In [ ]:
norm = ells*(ells+1)/(2*np.pi)
if 'GC' in use_obs :
    Nbins=Nbins_gc
    fig, axes = plt.subplots(ncols=3,  subplot_kw=dict(frameon=True), figsize=(21, 8))
    plt.suptitle('Relative difference between APS in GW-WL with old and new code', y=1.05, fontsize=20, fontweight='bold')
    
    axis_label_fontsize = 20
    title_fontsize = 20
    title_fontweight = 'bold'
    
    
    axes[0].set_title('GW-WL', loc='right', fontdict={'fontsize': title_fontsize, 'fontweight': title_fontweight})
    for i in range (1,10):
        axes[0].plot(ells, np.abs(calc_obs.Cls['W{}xW{}'.format(i, i)]-calc_obs2.Cls['WL{}xWL{}'.format(i, i)])/calc_obs2.Cls['WL{}xWL{}'.format(i, i)], label=r'$i={}$'.format(i),color=bincolors[i-1])
    #axes[2].plot(ells, calc_obs2.Cls['W{}xW{}'.format(10, 10)]*norm, label=r'$\mu-\Sigma$', color='blue', marker = '*')
    axes[0].set_xlabel(r'$\ell$', fontsize=axis_label_fontsize)
    axes[0].set_xscale('log')
    #axes[0].set_yscale('log')
    
  
    axes[1].set_title('GCph x GW-WL', loc='right', fontdict={'fontsize': title_fontsize, 'fontweight': title_fontweight})
    for i in range (1,10):
        axes[1].plot(ells, np.abs(calc_obs.Cls['G{}xW{}'.format(i, i)]-calc_obs2.Cls['G{}xWL{}'.format(i, i)])/calc_obs2.Cls['G{}xWL{}'.format(i, i)], label=r'$i={}$'.format(i),color=bincolors[i-1])
    #axes[4].plot(ells, calc_obs2.Cls['G{}xW{}'.format(10, 10)]*norm, label=r'$\mu-\Sigma$', color='blue', marker = '*')
    axes[1].set_xlabel(r'$\ell$', fontsize=axis_label_fontsize)
    axes[1].set_xscale('log')
    #axes[1].set_yscale('log')
    
    axes[2].set_title('WL x GW-WL', loc='right', fontdict={'fontsize': title_fontsize, 'fontweight': title_fontweight})
    for i in range (1,10):
        axes[2].plot(ells, np.abs(calc_obs.Cls['L{}xW{}'.format(i, i)]-calc_obs2.Cls['L{}xWL{}'.format(i, i)])/calc_obs2.Cls['L{}xWL{}'.format(i, i)], label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[2].set_xlabel(r'$\ell$', fontsize=axis_label_fontsize)
    axes[2].set_xscale('log')
    #axes[2].set_yscale('log')
    
    axes[0].set_ylabel(r'$\Delta C_{ii}^{AB}(\ell)[\%]$', fontsize=axis_label_fontsize)
    plt.subplots_adjust(hspace=1.2)
    axes[-1].legend(**sidelegend)
    
    

    #plt.savefig('Cls_relative_difference_old_new_GWWL.png', bbox_inches='tight');
    plt.show()

In [ ]:
norm = ells*(ells+1)/(2*np.pi)
fig, axes = plt.subplots(ncols=3, sharey=False, subplot_kw=dict(frameon=True),figsize=(12,4))
plt.suptitle('Fiducial Angular power spectra, old: (-), new: (*)',y=1.05)
axis_label_fontsize = 20
title_fontsize = 20
title_fontweight = 'bold'

    
if 'GWWL' in use_obs:
    axes[0].set_title('GWxWL',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_wl+1):
       
        axes[0].plot(ells,calc_obs2.Cls['WL{}xWL{}'.format(i,i)]*norm,
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[0].set_xlabel(r'$\ell$')
    axes[0].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[0].set_xscale('log')
    axes[0].set_yscale('log')


if 'GWC' in use_obs:
    axes[1].set_title('GWC',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_gw+1):
        
        axes[1].plot(ells,calc_obs2.Cls['WC{}xWC{}'.format(i,i)]*norm,
                     label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[1].set_xlabel(r'$\ell$')
    axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    
if 'GWC' in use_obs and 'GWWL' in use_obs:
    axes[2].set_title('GWCxGWWL',loc='right',fontdict={'fontsize':16})
    for i in range(1,Nbins_gc+1):
        axes[2].plot(ells,calc_obs2.Cls['WC{}xWL{}'.format(i,i)]*norm,
                    label=r'$i={}$'.format(i),color=bincolors[i-1])
    axes[2].set_xlabel(r'$\ell$')
    axes[2].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[2].set_xscale('log')
    axes[2].set_yscale('log')




axes[-1].legend(**sidelegend)
axes[0].set_ylabel(r'$\ell(\ell+1)/(2\pi)C_{ii}^{AB}(\ell)$', fontsize=axis_label_fontsize)
plt.subplots_adjust(hspace=1.2)
#plt.savefig('Cls_GWWL_old_new_test.png', bbox_inches='tight')
axes[-1].legend(**sidelegend);